Here we are going to generate a brain image concert where each pian note will change a roi in the brain

In [ ]:
### install dependencies
# !pip install pydub
# !pip install moviepy



In [ ]:
import argparse
import os

# imports
import sys

import ipynbname  # needed for notebook_path = ipynbname.path() to work
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm

PROJECT_ROOT = ipynbname.path().parent.parent.parent
sys.path.append(str(PROJECT_ROOT))




# Image Stuff --------------------
from moviepy import ImageClip, concatenate_videoclips

import configs as cfg
import generation as generation
import src.brainst_img.instantiate_models as instantiate_brainst_img
import src.brainst_vol.instantiate_models as instantiate_brainst_vol
import src.utils.functions as fc
import src.utils.nifti_functions as nfc
import src.utils.util_freesurfer_segmentation as ufs
from src.brainst_img import generate_image, null_inversion, utils_generation
from src.utils import data_normalization, prep_segmentation, prep_volumes

# configs
PATH_BASE = ipynbname.path().parent
PATH_DATA = os.path.join(PATH_BASE, "data")
PATH_TEMP = os.path.join(PATH_DATA, "__temp__")
PATH_INPUT = os.path.join(PATH_DATA, "inputs")
PATH_OUTPUT = os.path.join(PATH_DATA, "outputs")
PATH_OUTPUT_SYNTHESIS = os.path.join(PATH_OUTPUT, "synthesis")
PATH_OUTPUT_LONGITUDINAL = os.path.join(PATH_OUTPUT, "longitudinal")
PATH_OUTPUT_VIDEOS = os.path.join(PATH_OUTPUT, "videos")
PATH_OUTPUT_NULL_INVERSION = os.path.join(PATH_OUTPUT, "null_inversion")

os.makedirs(PATH_OUTPUT_SYNTHESIS, exist_ok=True)
os.makedirs(PATH_OUTPUT_LONGITUDINAL, exist_ok=True)
os.makedirs(PATH_OUTPUT_VIDEOS, exist_ok=True)
os.makedirs(PATH_OUTPUT_NULL_INVERSION, exist_ok=True)

SEED = 2
BRAINST_IMG_DIFUSION_STEPS = 50
BRAINST_VOL_DIFUSION_STEPS = 50

BPS = 2 # beats per second (bigger is faster)
FPS = 24

USED_AXIAL_LAYER = cfg.SHAPE_PREP_IMG[2]//2
USED_CORONAL_LAYER = cfg.SHAPE_PREP_IMG[1]//2
USED_SAGITTAL_LAYER = cfg.SHAPE_PREP_IMG[0]//2 -15


Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work


# Base image generation (Synthesis)

### Core functions copied from main_generation.py and generation.py

In [3]:
def instantiate_brainst_img_model(diffusion_steps: int = 50) -> dict:
    """Instantiate the BrainST-img model bundle from ``configs.py``.

    Args:
        diffusion_steps: Number of denoising steps to configure the noise
            scheduler with.

    Returns:
        Model bundle containing (at least) ``unet``, ``conditions_model``,
        ``noise_scheduler``, and ``autoencoder``.
    """
    networks_config = fc.dict_to_args(cfg.ARCHITECTURE_BRAINST_IMG, deep_conversion=True)
    brainst_img = instantiate_brainst_img.instantiate_model_and_load(
        networks_config,
        cfg.PATH_BRAINST_IMG_CHK,
        cfg.PATH_AUTOENCODER_CHK,
        device=cfg.DEVICE,
        dm_num_inference_steps=diffusion_steps,
    )
    return brainst_img


def instantiate_brainst_vol_model(diffusion_steps: int = 50) -> dict:
    """Instantiate the BrainST-vol model bundle from ``configs.py``.

    Args:
        diffusion_steps: Number of denoising steps to configure the noise
            scheduler with.

    Returns:
        Model bundle containing (at least) ``unet``, ``conditions_model``,
        and ``noise_scheduler``.
    """
    networks_config = fc.dict_to_args(cfg.ARCHITECTURE_BRAINST_VOL, deep_conversion=True)
    brainst_vol = instantiate_brainst_vol.instantiate_model_and_load(
        networks_config, cfg.PATH_BRAINST_VOL_CHK, device=cfg.DEVICE, dm_num_inference_steps=diffusion_steps
    )
    return brainst_vol



# main_generation.py
def create_covariates_dict(args: argparse.Namespace, normalizer=None, is_target: bool = False) -> dict:
    """Build the ``{age, sex, dx}`` covariates dict expected by the models.

    Args:
        args: Parsed arguments containing either the ``initial_*`` or
            ``target_*`` covariate fields.
        normalizer: Fitted normalizer used to standardize the raw age
            value.
        is_target: If True, build the covariates dict from the
            ``target_*`` fields; otherwise use the ``initial_*`` fields.

    Returns:
        Dictionary with normalized ``age`` and integer-coded ``sex``/``dx``.
    """
    prefix = "target" if is_target else "initial"
    return {
        "age": normalizer.transform_single(getattr(args, f"{prefix}_age"), "age"),
        "sex": cfg.SEX_MAPPING[getattr(args, f"{prefix}_sex")],
        "dx": cfg.DX_MAPPING[getattr(args, f"{prefix}_dx")],
    }

### Generate images (and intermediate steps)

In [4]:
base_img_covariates_dict = {
    "target_age": 70,
    "target_sex": 'M',
    "target_dx": 'CN'
}

# ---- 1. Generate brain structure volumes with BrainST-vol
normalizer = data_normalization.SavedNormalizerBrainStructures(cfg.PATH_NORMALIZATION_PARAMS)
brainst_vol = instantiate_brainst_vol_model(diffusion_steps=BRAINST_VOL_DIFUSION_STEPS)

base_img_covariates_normalized = create_covariates_dict(fc.dict_to_args(base_img_covariates_dict, deep_conversion=True), normalizer=normalizer, is_target=True)

target_roi_volumes_dict = generation.brainst_vol_synthesis(
    brainst_vol, base_img_covariates_normalized, seed=SEED, free_guidance_ratio=cfg.BRAINST_VOL_FREE_GUIDANCE_RATIO
)

# ---- 2. Generate images with BrainST-img
brainst_img = instantiate_brainst_img_model(diffusion_steps=BRAINST_IMG_DIFUSION_STEPS)
device = next(brainst_img["unet"].parameters()).device

# 2.1 Sample the initial Gaussian noise latents the diffusion process starts from.
noisy_latents = utils_generation.gen_random_latents(cfg.SHAPE_LATENT, seed=SEED, device=device).unsqueeze(0)

# 2.2 Run the reverse diffusion process and decode straight to image space.
reconstructed_latents = generate_image.diffusion_loop(
    noisy_latents=noisy_latents,
    unet=brainst_img["unet"],
    conditions_model=brainst_img["conditions_model"],
    noise_scheduler=brainst_img["noise_scheduler"],
    autoencoder=brainst_img["autoencoder"],
    conditions_list=[target_roi_volumes_dict],
    conditions_keys_ordered=cfg.STRUCTURE_NAME_LIST_VOL,
    uncond_embeddings=None,
    free_guidance_ratio=cfg.BRAINST_IMG_FREE_GUIDANCE_RATIO,
    decode_img=True,
    decode_first=True,
    decode_complete=True,
    return_denoising_steps=True
)

path_name_base_img = os.path.join(PATH_OUTPUT_SYNTHESIS, f"img_{base_img_covariates_dict['target_age']}_{base_img_covariates_dict['target_sex']}_{base_img_covariates_dict['target_dx']}.nii.gz")
img_base = reconstructed_latents["images"][0].astype(np.float32)
nfc.save_nifti(img_base, None, path_name_base_img)

# segment the base image
path_name_base_seg = os.path.join(PATH_OUTPUT_SYNTHESIS, f"seg_{base_img_covariates_dict['target_age']}_{base_img_covariates_dict['target_sex']}_{base_img_covariates_dict['target_dx']}.nii.gz")
prep_segmentation.save_synthseg_segmentation(path_name_base_img, path_name_base_seg, verify=True)

seg_base = nfc.load_nifti(path_name_base_seg)[0]
seg_base3 = ufs.merge_seg96_to_seg3(seg_base)


# 3. Extract the decoded image (batch of 1).
# reconstructed_img = reconstructed_latents["images"][0].astype(np.float32)

#F: {'total_vol': -0.3974604606628418, 'surrounding_csf_vol': -0.3007447421550751, 'cortical_gm_vol': -0.5264509916305542, 'cerebral_wm_vol': 0.8769873976707458, 'lateral_ventricles_vol': -0.9943557977676392, 'third_ventricle_vol': -1.4352394342422485, 'fourth_ventricle_vol': 0.9504517912864685, 'thalamus_vol': 1.9734870195388794, 'hippocampus_vol': 0.7973410487174988, 'amygdala_vol': 1.6776809692382812, 'putamen_vol': 1.938072681427002, 'pallidum_vol': 3.2328848838806152, 'caudate_vol': 1.7349056005477905, 'accumbens_area_vol': 1.985140085220337, 'ventral_dc_vol': 1.370397686958313, 'cerebellum_gm_vol': 1.5843052864074707, 'cerebellum_wm_vol': 0.005883472505956888, 'brainstem_vol': -0.18310894072055817}
#M: {'total_vol': -0.2272251546382904, 'surrounding_csf_vol': 0.019221236929297447, 'cortical_gm_vol': -0.63700932264328, 'cerebral_wm_vol': 0.7733215093612671, 'lateral_ventricles_vol': -0.7830736637115479, 'third_ventricle_vol': -0.9731443524360657, 'fourth_ventricle_vol': 0.8755730986595154, 'thalamus_vol': 1.3446475267410278, 'hippocampus_vol': 0.4009440541267395, 'amygdala_vol': 1.462445616722107, 'putamen_vol': 1.4184178113937378, 'pallidum_vol': 2.7166476249694824, 'caudate_vol': 0.9138857126235962, 'accumbens_area_vol': 1.5280821323394775, 'ventral_dc_vol': 0.9571276903152466, 'cerebellum_gm_vol': 1.1415350437164307, 'cerebellum_wm_vol': -0.2714678943157196, 'brainstem_vol': -0.3698115348815918}

100%|██████████| 50/50 [00:02<00:00, 19.66it/s]


### Decode intermediate steps

In [5]:
decoded_denoising_steps = []
bar = tqdm(total=len(reconstructed_latents["denoising_steps"]), desc="Decoding denoising steps")
for i, latent_step in enumerate(reconstructed_latents["denoising_steps"]):
    path_name_base_img_step = os.path.join(PATH_OUTPUT_SYNTHESIS, f"img_{base_img_covariates_dict['target_age']}_{base_img_covariates_dict['target_sex']}_{base_img_covariates_dict['target_dx']}_step_{i}.nii.gz")
    if not os.path.exists(path_name_base_img_step):
        # decode 
        latent_step = torch.from_numpy(latent_step).to(device)
        decoded_step = brainst_img["autoencoder"].decode(latent_step).squeeze().cpu().numpy()
        decoded_step = np.clip(decoded_step, 0, 1)  # Clip values to [0, 1] range

        # save
        nfc.save_nifti(decoded_step.astype(np.float32), None, path_name_base_img_step)
    else:
        # load
        decoded_step = nfc.load_nifti(path_name_base_img_step)[0]
    decoded_denoising_steps.append(decoded_step)
    bar.update(1)
bar.close()

Decoding denoising steps: 100%|██████████| 51/51 [00:11<00:00,  4.61it/s]


### Show results

In [6]:
# img_steps_2D_list = [data_normalization.normalize_image(_img)[...,USED_AXIAL_LAYER] for _img in decoded_denoising_steps]
# img_base_2D = data_normalization.normalize_image(reconstructed_latents["images"][0])[...,USED_AXIAL_LAYER]

# img_base_2D = data_normalization.normalize_image(img_base)[...,USED_AXIAL_LAYER]
# seg_base3_2D = data_normalization.normalize_image(seg_base3)[...,USED_AXIAL_LAYER]

# fc.imgshow_list([img_base_2D, seg_base3_2D]+img_steps_2D_list, name="Base image")


### Synthesis video

In [7]:
def get_2D_slice(img_data, to_rgb=True, new_h=None, nb_views=2, axial_layer=USED_AXIAL_LAYER, coronal_layer=USED_CORONAL_LAYER, sagittal_layer=USED_SAGITTAL_LAYER):
    axial_offset = abs(img_data.shape[2]//2 - axial_layer)
    coronal_offset = abs(img_data.shape[1]//2 - coronal_layer)
    sagittal_offset = abs(img_data.shape[0]//2 - sagittal_layer)

    if nb_views == 3:
        img_2D = fc.cat_n_views_different_layers([img_data], axis=1, view_layersoffset_list=((2,axial_offset), (1,coronal_offset), (0,sagittal_offset)), img_cropping=0, to_rgb=to_rgb)[0]
    elif nb_views == 2:
        img_2D = fc.cat_n_views_different_layers([img_data], axis=1, view_layersoffset_list=((2,axial_offset), (0,sagittal_offset)), img_cropping=0, to_rgb=to_rgb)[0]
    else:
        img_2D = fc.cat_n_views_different_layers([img_data], axis=1, view_layersoffset_list=((2,axial_offset)), img_cropping=0, to_rgb=to_rgb)[0]

    if new_h is not None:
        img_2D = fc.resize_image(img_2D, new_h, mode='h')
    return img_2D

In [8]:

# ----- Create image video
frames = []
time = 0.25 # ich image will last a sexteen
synthesis_video_img_list = decoded_denoising_steps + [img_base]*4
bar = tqdm(total=len(synthesis_video_img_list), desc="Creating video frames")
for img_step in synthesis_video_img_list:
    # images
    img = get_2D_slice(img_step, to_rgb=True, new_h=512)
    frames.append((img, time/BPS))
    bar.update(1)
bar.close()


clips = [ImageClip(img).with_duration(d) for img, d in frames]
final_clip = concatenate_videoclips(clips, method="compose")

video_name = f"synthesis_creation_{base_img_covariates_dict['target_age']}_{base_img_covariates_dict['target_sex']}_{base_img_covariates_dict['target_dx']}.mp4"
video_path_name = os.path.join(PATH_OUTPUT_VIDEOS, video_name)
final_clip.write_videofile(video_path_name, fps=FPS, logger=None)
print(f"Exported video to: {os.path.basename(video_path_name)}")
    

Creating video frames: 100%|██████████| 55/55 [00:10<00:00,  5.30it/s]


Exported video to: synthesis_creation_70_M_CN.mp4


# Longitudinal generation

In [9]:
def get_volumes_from_segmentation(seg: np.ndarray, normalizer) -> dict:
    """Compute a normalized ROI-volumes dict directly from a segmentation map.

    Pipeline: raw voxel counts per ROI -> percentage of intracranial
    volume (ICV) -> standardized (z-scored) scale used by the models.

    Args:
        seg: Segmentation label map (one integer label per anatomical structure).
        normalizer: Fitted normalizer used for the final standardization step.

    Returns:
        ``{roi_name: standardized_volume}`` for every ROI in
        ``cfg.STRUCTURE_NAME_LIST_VOL``.
    """
    roi_volumes_dict = prep_volumes.get_volumes(seg, cfg.STRUCTURE_INDEX_VOL_DICT)
    roi_volumes_dict = data_normalization.normalize_by_icv(
        pd.DataFrame([roi_volumes_dict]),
        structure_names=cfg.STRUCTURE_NAME_LIST_VOL,
        icv_column="total_vol",
        percentage=False,
    ).iloc[0].to_dict()
    roi_volumes_dict = normalizer.transform(pd.DataFrame([roi_volumes_dict])).iloc[0].to_dict()

    return roi_volumes_dict

In [10]:
# obtain initial latents
initial_latents = brainst_img["autoencoder"].encode(img_base).cpu().numpy()

# create output paths for latents and unconditional embeddings
latents_output_path_name = os.path.join(PATH_OUTPUT_NULL_INVERSION, f"latents_{base_img_covariates_dict['target_age']}_{base_img_covariates_dict['target_sex']}_{base_img_covariates_dict['target_dx']}.npy")
uncond_embeddings_path_name = os.path.join(PATH_OUTPUT_NULL_INVERSION, f"uncond_embeddings_{base_img_covariates_dict['target_age']}_{base_img_covariates_dict['target_sex']}_{base_img_covariates_dict['target_dx']}.npy")

# 1. Compute the source image's current ROI-volumes profile.
initial_roi_volumes_dict = get_volumes_from_segmentation(seg_base, normalizer)
print("Initial covariate dict: %s", base_img_covariates_dict)
print("Initial covariate dict normalized: %s", base_img_covariates_normalized)
print("Initial ROI volumes dict: %s", initial_roi_volumes_dict)

age_gap = 20
ages_list = [base_img_covariates_dict['target_age'] + i for i in range(-age_gap, age_gap+1, 1)]

longitudinal_img_list = []
bar = tqdm(total=len(ages_list), desc="Generating longitudinal images")
for age in ages_list:
    bar.set_description(f"Generating age: {age}")

    img_name = f"longitudinal_img_{age}_{base_img_covariates_dict['target_sex']}_{base_img_covariates_dict['target_dx']}.nii.gz"
    img_path_name = os.path.join(PATH_OUTPUT_LONGITUDINAL, img_name)

    if not os.path.exists(img_path_name):
        # print(f"Generating longitudinal image for age: {age}")
        # 2. Predict the target ROI-volumes profile by transforming the
        longitudinal_img_covariates_dict = base_img_covariates_dict.copy()
        longitudinal_img_covariates_dict["target_age"] = age

        longitudinal_img_covariates_normalized = create_covariates_dict(fc.dict_to_args(longitudinal_img_covariates_dict, deep_conversion=True), normalizer=normalizer, is_target=True)
        target_roi_volumes_dict = generation.brainst_vol_transformation(
            brainst_vol, initial_roi_volumes_dict, base_img_covariates_normalized, longitudinal_img_covariates_normalized
        )
        # print("Target covariate dict: %s", longitudinal_img_covariates_dict)
        # print("Target covariate dict normalized: %s", longitudinal_img_covariates_normalized)
        # print("Target ROI volumes dict: %s", target_roi_volumes_dict)

        # 3.2 Invert: recover the noisy latents + null-text embeddings that
        inversion_result = null_inversion.create_save_load_null_inversion_results(
            brainst_img["unet"],
            brainst_img["conditions_model"],
            brainst_img["noise_scheduler"],
            initial_latents,
            initial_roi_volumes_dict,
            cfg.STRUCTURE_NAME_LIST_VOL,
            free_guidance_ratio=cfg.BRAINST_IMG_FREE_GUIDANCE_RATIO,
            compute_uncond_embeddings=True,
            num_inner_steps=2,
            early_stop_epsilon=1e-8,
            verbose=False,
            latents_output_path_name=latents_output_path_name,
            uncond_embeddings_path_name=uncond_embeddings_path_name,
        )

        # 3.3 Run the reverse diffusion process from the inverted (noisiest)
        reconstructed_latents = generate_image.diffusion_loop(
            noisy_latents=inversion_result["noisy_latents"],
            unet=brainst_img["unet"],
            conditions_model=brainst_img["conditions_model"],
            noise_scheduler=brainst_img["noise_scheduler"],
            autoencoder=brainst_img["autoencoder"],
            conditions_list=[target_roi_volumes_dict],
            conditions_keys_ordered=cfg.STRUCTURE_NAME_LIST_VOL,
            uncond_embeddings=inversion_result["uncond_embeddings"],
            free_guidance_ratio=cfg.BRAINST_IMG_FREE_GUIDANCE_RATIO,
            decode_img=True,
            decode_first=True,
            decode_complete=True,
        )

        # save the image
        img_longitudinal = reconstructed_latents["images"][0].astype(np.float32)
        nfc.save_nifti(img_longitudinal, None, img_path_name)
    else:
        # print(f"Longitudinal image for age {age} already exists. Loading from disk.")
        img_longitudinal = nfc.load_nifti(img_path_name)[0]
    longitudinal_img_list.append(img_longitudinal)
    bar.update(1)
bar.close()




Initial covariate dict: %s {'target_age': 70, 'target_sex': 'M', 'target_dx': 'CN'}
Initial covariate dict normalized: %s {'age': -0.5880268731788723, 'sex': 1, 'dx': 0}
Initial ROI volumes dict: %s {'total_vol': -0.31791875795071334, 'surrounding_csf_vol': 0.07366247486962935, 'cortical_gm_vol': -1.3486683540214242, 'cerebral_wm_vol': 0.8320968775277999, 'lateral_ventricles_vol': -0.700533746728846, 'third_ventricle_vol': -0.9179586305259683, 'fourth_ventricle_vol': 1.2858923602524823, 'thalamus_vol': 1.1453062794985949, 'hippocampus_vol': -0.16107329872838105, 'amygdala_vol': 1.4430161732998963, 'putamen_vol': 1.7908458822953102, 'pallidum_vol': 2.357391423908243, 'caudate_vol': 0.7906417327124162, 'accumbens_area_vol': 1.2478902489492383, 'ventral_dc_vol': 0.5078665625128712, 'cerebellum_gm_vol': 1.609386154538152, 'cerebellum_wm_vol': -0.20694882867142106, 'brainstem_vol': -0.26388743863076886}


Generating age: 90: 100%|██████████| 41/41 [00:09<00:00,  4.21it/s]   


### Longitudinal video

In [11]:

# ----- Create image video
frames = []
time = 0.5 # each image will last a beat

# # forward longitudinal video
# video_name = f"longitudinal_fordward_{base_img_covariates_dict['target_age']}_{base_img_covariates_dict['target_sex']}_{base_img_covariates_dict['target_dx']}.mp4"
# longitudinal_img_video_list = longitudinal_img_list

# # starts with the img_base, then go to the lowest age, then do the normal longitidunal
video_name = f"longitudinal_center_left_right_{base_img_covariates_dict['target_age']}_{base_img_covariates_dict['target_sex']}_{base_img_covariates_dict['target_dx']}.mp4"
# center_img = longitudinal_img_list[(nb_imgs//2)+1]
center_img = img_base
nb_imgs = len(longitudinal_img_list)
longitudinal_img_video_list = [center_img]*2 + longitudinal_img_list[:nb_imgs//2][::-1] + longitudinal_img_list

bar = tqdm(total=len(longitudinal_img_video_list), desc="Creating video frames")

for img_step in longitudinal_img_video_list:
    img = get_2D_slice(img_step, to_rgb=True, new_h=512)
    frames.append((img, time/BPS))
    bar.update(1)
bar.close()


clips = [ImageClip(img).with_duration(d) for img, d in frames]
final_clip = concatenate_videoclips(clips, method="compose")

video_path_name = os.path.join(PATH_OUTPUT_VIDEOS, video_name)
final_clip.write_videofile(video_path_name, fps=FPS, logger=None)
print(f"Exported video to: {os.path.basename(video_path_name)}")
    

Creating video frames: 100%|██████████| 63/63 [00:11<00:00,  5.40it/s]


Exported video to: longitudinal_center_left_right_70_M_CN.mp4
